# Online Memory Degradation Detector

This notebook implements a dynamic programming approach to evaluate whether a differentiable stack model has learned an exact algorithm or a length-dependent shortcut. 

It analyzes the empirical error probabilities $\epsilon_t$ at each step during an Out-Of-Distribution (OOD) rollout. By treating the stack trajectories as a Poisson Binomial distribution, we can calculate cumulative variance and establish a real-time theoretical upper bound on memory fidelity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def evaluate_algorithmic_learning(epsilons_t, threshold=0.5):
    """
    Evaluates algorithmic learning based on epsilon drift and cumulative variance.
    
    Args:
        epsilons_t: 1D numpy array of the error probability at each step during OOD.
        threshold: The probability where we consider retrieval 'random guessing' (e.g. 0.5).
        
    Returns:
        is_shortcut (bool): True if significant OOD epsilon drift is detected.
        fidelity_bounds (np.array): Theoretical max fidelity bound at each step.
        failure_step (int/str): The exact step where fidelity drops below the threshold.
    """
    # 1. Test for Invariant Algorithm vs Shortcut (Drift)
    # If the model learned a shortcut, the hidden state will drift OOD, causing epsilon to rise.
    first_half_mean = np.mean(epsilons_t[:len(epsilons_t)//2])
    second_half_mean = np.mean(epsilons_t[len(epsilons_t)//2:])
    
    # Simple heuristic for drift: average error rate doubles in the second half
    is_shortcut = second_half_mean > (first_half_mean * 2) 
    
    # 2. Dynamic Programming Bound (The Poisson Binomial approach)
    # Calculate step-wise variance: Var(X_t) = p * (1-p)
    step_variances = epsilons_t * (1 - epsilons_t)
    
    # Cumulative variance over the token's lifespan on the stack
    cumulative_variance = np.cumsum(step_variances)
    
    # Calculate the theoretical maximum fidelity bound at each step
    # Using the Baillon universal bound for Poisson Binomials
    e = np.exp(1)
    fidelity_bounds = 1.0 / np.sqrt(2 * e * cumulative_variance + 1e-9) # 1e-9 avoids division by zero
    fidelity_bounds = np.clip(fidelity_bounds, 0, 1) # Bound probabilities to max 1
    
    # Find the timestep where theoretical fidelity drops below random guessing
    failure_indices = np.where(fidelity_bounds < threshold)[0]
    failure_step = failure_indices[0] if len(failure_indices) > 0 else "Never"

    return is_shortcut, fidelity_bounds, failure_step

In [ ]:
# --- Example Usage & Visualization ---

# 1. Load your actual data here. 
# e.g., epsilons_long = np.load('StackRNN/results/reversal/baseline_new/epsilons500.npy')

# For demonstration, we'll simulate a model that learned a shortcut (drifting epsilon)
np.random.seed(42)
seq_length = 200
dummy_epsilons = np.linspace(0.01, 0.12, seq_length) + np.random.normal(0, 0.005, seq_length)
epsilons_long = np.clip(dummy_epsilons, 0.0, 0.5)

# 2. Run the evaluation
is_shortcut, bounds, failure_step = evaluate_algorithmic_learning(epsilons_long)

print(f"Shortcut Detected (Epsilon Drift): {is_shortcut}")
print(f"Predicted Failure Step (Fidelity < 0.5): {failure_step}")

# 3. Plot the results
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot empirical epsilon_t
color = 'tab:red'
ax1.set_xlabel('Timestep (t)')
ax1.set_ylabel('Error Probability (ε_t)', color=color)
ax1.plot(epsilons_long, color=color, alpha=0.7, label='Empirical ε_t')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 0.5)

# Instantiate a second axes that shares the same x-axis
ax2 = ax1.twinx()  

# Plot the theoretical bounds
color = 'tab:blue'
ax2.set_ylabel('Theoretical Fidelity Bound', color=color)  
ax2.plot(bounds, color=color, linestyle='--', linewidth=2, label='Baillon Fidelity Bound')
ax2.axhline(0.5, color='gray', linestyle=':', label='Random Guessing (0.5)')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0, 1.05)

# Mark the predicted failure step
if failure_step != "Never":
    ax2.axvline(failure_step, color='black', linestyle='-.', alpha=0.5, label=f'Theoretical Failure (t={failure_step})')

fig.tight_layout()  
plt.title('Memory Degradation: ε_t Drift vs. Theoretical Fidelity')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.85))

plt.show()